In [1]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

# FFHQ: HDRI_sota_sj.json

In [ ]:
ax = 2
start_c = 0.5
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/Video_supp/ffhq_sup_video_rot{ax}_{start_c}C.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/selected_rotate_RT_60065.json"
sampling_rate = 2
def reduce_sampling_rate(frame_list, sampling_rate=1):
    # Reduce sampling rate and include first and last
    reduced_frame_list = frame_list[::sampling_rate]  # Take every nth frame
    if frame_list[0] not in reduced_frame_list:
        print("Adding first frame...")
        reduced_frame_list.insert(0, frame_list[0])  # Ensure the first frame is included
    if frame_list[-1] not in reduced_frame_list:
        print("Adding last frame...")
        reduced_frame_list.append(frame_list[-1])  # Ensure the last frame is included
    return reduced_frame_list

# The results were generated following the order of 
# rotate@{start_c}C -> diffuse (0.0) -> shadow (1.0) -> rotate@1.0C
method = [f"ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C", 
          f"ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C", 
          f"ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C", 
          f"ours_difareli++_oneshot_rot{ax}_dstL_maxC",
        ]
os.makedirs("./vids/", exist_ok=True)
os.makedirs(f'./vids/all_outputs/res/', exist_ok=True)
os.makedirs(f'./vids/all_outputs/misc/', exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    fidx = int(60 * 0.95)
    for i, m in enumerate(method):
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/', exist_ok=True)
        img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
        relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        
        # Copy all images to a new folder
        for img in relit:
            ii = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{ii:04d}.png')
    
    for i, m in enumerate(method):
        if i == 0:  # Rotate @ certain C (0.6)
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            start_offset = 20
            start_idx = fidx - start_offset
            relit = relit[start_idx:] + relit[:fidx]

            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
                    
            # High quality video crf 17 with 24 fps (-y to overwrite)
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4', 
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_rev.mp4 -y'
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
                    
        elif i == 1:    # Diffuse from 0.6 to 0.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = reduce_sampling_rate(relit, sampling_rate)
            print(len(relit))
            assert False
            
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
            filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
            # High quality video crf 17 with 24 fps (-y to overwrite), its reversed version and concat them 
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}.mp4', 
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -y -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4 {filter_cat} -y ./vids/axis={ax}/{src}_{dst}/{m}_out.mp4']
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
             
        elif i == 2:    # Shadow from 0.6 to 1.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = reduce_sampling_rate(relit, sampling_rate)
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
            # High quality video crf 17 with 24 fps (-y to overwrite), its reversed version and concat them 
            filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}.mp4', 
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -y -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4',
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
        elif i == 3:    # Rotate @ 1.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = relit[fidx:] + relit[:fidx]
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
            # High quality video crf 17 with 24 fps (-y to overwrite)
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4', 
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_rev.mp4 -y'
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
        else:
            raise ValueError("Unknown method...")
    # Concat all videos together
    filter_cat = f'-filter_complex "[0:v][1:v][2:v][3:v][4:v][5:v][6:v]concat=n=7:v=1:a=0"'
    cmd = f"""
        ffmpeg 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C_out.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_rev.mp4 
            {filter_cat} 
            -y ./vids/all_outputs/res/{src}_{dst}_axis={ax}.mp4
    """
    # Remove extra spaces and newlines
    cmd = " ".join(cmd.strip().split())
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError:
        print(f"Error in {cmd}")
    assert False

  0%|          | 0/214 [00:00<?, ?it/s]

  0%|          | 0/214 [00:01<?, ?it/s]

30


AssertionError: 

# Get Shading reference + Shadow map

In [11]:
ax = 2
start_c = 0.5
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/Video_supp/ffhq_sup_video_rot{ax}_{start_c}C.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/selected_rotate_RT_60065.json"
sampling_rate = 2
def reduce_sampling_rate(frame_list, sampling_rate=1):
    # Reduce sampling rate and include first and last
    reduced_frame_list = frame_list[::sampling_rate]  # Take every nth frame
    if frame_list[0] not in reduced_frame_list:
        reduced_frame_list.insert(0, frame_list[0])  # Ensure the first frame is included
    if frame_list[-1] not in reduced_frame_list:
        reduced_frame_list.append(frame_list[-1])  # Ensure the last frame is included
    return reduced_frame_list

# The results were generated following the order of 
# rotate@{start_c}C -> diffuse (0.0) -> shadow (1.0) -> rotate@1.0C
method = [f"ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C", 
          f"ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C", 
          f"ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C", 
          f"ours_difareli++_oneshot_rot{ax}_dstL_maxC",
        ]
os.makedirs("./vids/", exist_ok=True)
os.makedirs(f'./vids/all_outputs/res/', exist_ok=True)
os.makedirs(f'./vids/all_outputs/misc/', exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    fidx = int(60 * 0.95)
    for i, m in enumerate(method):
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/', exist_ok=True)
        img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
        relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        # Copy all images to a new folder
        for img in relit:
            ii = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{ii:04d}.png')
        
        shadow = sort_by_frame(glob.glob(f'{img_path}/dst_shadm_shad_f*.png'))[1:]
        render = sort_by_frame(glob.glob(f'{img_path}/dst_ren_f*.png'))[1:]
        # Copy all images to a new folder
        for img in shadow:
            ii = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/shadow_frame_{ii:04d}.png')
        for img in render:
            ii = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/render_frame_{ii:04d}.png')
        
    
    for i, m in enumerate(method):
        if i == 0:  # Rotate @ certain C (0.6)
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            render = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/render_frame_*.png'))
            shadow = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/shadow_frame_*.png'))
            start_offset = 20
            start_idx = fidx - start_offset
            relit = relit[start_idx:] + relit[:fidx]
            render = render[start_idx:] + render[:fidx]
            shadow = shadow[start_idx:] + shadow[:fidx]

            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
                for j, img in enumerate(render):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_{j:04d}.png')
                for j, img in enumerate(shadow):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_{j:04d}.png')
                    
                    
            # High quality video crf 17 with 24 fps (-y to overwrite)
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4', 
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_rev.mp4 -y',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render_rev.mp4 -y',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow_rev.mp4 -y',
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
                    
        elif i == 1:    # Diffuse from 0.6 to 0.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = reduce_sampling_rate(relit, sampling_rate)
            render = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/render_frame_*.png'))
            render = reduce_sampling_rate(render, sampling_rate)
            shadow = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/shadow_frame_*.png'))
            shadow = reduce_sampling_rate(shadow, sampling_rate)
            
            
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
                for j, img in enumerate(render):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_{j:04d}.png')
                for j, img in enumerate(shadow):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_{j:04d}.png')    

            filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
            # High quality video crf 17 with 24 fps (-y to overwrite), its reversed version and concat them 
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}.mp4', 
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -y -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4 {filter_cat} -y ./vids/axis={ax}/{src}_{dst}/{m}_out.mp4',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_render.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_render.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_render_rev.mp4 -y',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_render.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}_render_rev.mp4 {filter_cat} -y ./vids/axis={ax}/{src}_{dst}/{m}_render_out.mp4',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_shadow.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_shadow.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_shadow_rev.mp4 -y',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_shadow.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}_shadow_rev.mp4 {filter_cat} -y ./vids/axis={ax}/{src}_{dst}/{m}_shadow_out.mp4',]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
             
        elif i == 2:    # Shadow from 0.6 to 1.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = reduce_sampling_rate(relit, sampling_rate)
            render = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/render_frame_*.png'))
            render = reduce_sampling_rate(render, sampling_rate)
            shadow = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/shadow_frame_*.png'))
            shadow = reduce_sampling_rate(shadow, sampling_rate)
            
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
                for j, img in enumerate(render):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_{j:04d}.png')
                for j, img in enumerate(shadow):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_{j:04d}.png')
            # High quality video crf 17 with 24 fps (-y to overwrite), its reversed version and concat them 
            filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}.mp4', 
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}.mp4 -y -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_rev.mp4',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_render.mp4',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_render.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_render_rev.mp4 -y',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_shadow.mp4',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_shadow.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_shadow_rev.mp4 -y',
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
        elif i == 3:    # Rotate @ 1.0
            relit = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_*.png'))
            relit = relit[fidx:] + relit[:fidx]
            render = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/render_frame_*.png'))
            render = render[fidx:] + render[:fidx]
            shadow = sorted(glob.glob(f'./vids/axis={ax}/{src}_{dst}/{m}/shadow_frame_*.png'))
            shadow = shadow[fidx:] + shadow[:fidx]
            # Copy the reindexed images to same folder but with new names
            with open(f'./vids/axis={ax}/{src}_{dst}/{m}_reidx/frame_order.txt', 'w') as f:
                for j, img in enumerate(relit):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_{j:04d}.png')
                    f.write(f'{org_idx} -> {j}\n')
                for j, img in enumerate(render):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_{j:04d}.png')
                for j, img in enumerate(shadow):
                    org_idx = int(img.split('/')[-1].split('_')[-1].split('.')[0])
                    os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_{j:04d}.png')
                
            # High quality video crf 17 with 24 fps (-y to overwrite)
            cmd = [f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4', 
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_rev.mp4 -y',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/render_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_render_rev.mp4 -y',
                   f'ffmpeg -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx/shadow_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow.mp4',
                   f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}_reidx_shadow_rev.mp4 -y',
                ]
            for c in cmd:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError:
                    print(f"Error in {c}")
        else:
            raise ValueError("Unknown method...")
    # Concat all videos together
    filter_cat = f'-filter_complex "[0:v][1:v][2:v][3:v][4:v][5:v][6:v]concat=n=7:v=1:a=0"'
    # hstack
    filter_hstack = f'-filter_complex "[0:v][1:v][2:v]hstack=3[v]" -map "[v]"'
    cmd = [
        f"""
        ffmpeg 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C_out.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_rev.mp4 
            {filter_cat} 
            -y ./vids/all_outputs/res/{src}_{dst}_axis={ax}.mp4""", 
        f"""
        ffmpeg
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_render.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C_render_out.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_render.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_render.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_render_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_render_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_render_rev.mp4 
            {filter_cat} 
            -y ./vids/all_outputs/misc/{src}_{dst}_axis={ax}_render.mp4""",
        f"""
        ffmpeg
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_shadow.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to0.0C_shadow_out.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_shadow.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_shadow.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_maxC_reidx_shadow_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_reshadow_rot{ax}_{start_c}to1.0C_shadow_rev.mp4 
            -i ./vids/axis={ax}/{src}_{dst}/ours_difareli++_oneshot_rot{ax}_dstL_{start_c}C_reidx_shadow_rev.mp4 
            {filter_cat} 
            -y ./vids/all_outputs/misc/{src}_{dst}_axis={ax}_shadow.mp4""",
        f"""
        ffmpeg
            -i ./vids/all_outputs/misc/{src}_{dst}_axis={ax}_render.mp4
            -i ./vids/all_outputs/res/{src}_{dst}_axis={ax}.mp4
            -i ./vids/all_outputs/misc/{src}_{dst}_axis={ax}_shadow.mp4
            {filter_hstack}
            -y ./vids/all_outputs/res/{src}_{dst}_axis={ax}_combined.mp4
        """
        ]
    # Remove extra spaces and newlines
    for c in cmd:
        c = " ".join(c.strip().split())
        try:
            subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError:
            print(f"Error in {c}")

100%|██████████| 214/214 [31:06<00:00,  8.72s/it]
